Leitura das tabelas da camada Silver

In [0]:
df_passagens = spark.table(
    "viagens_teste.silver.passagens"
)

df_hospedagens = spark.table(
    "viagens_teste.silver.hospedagens"
)

df_uber = spark.table(
    "viagens_teste.silver.uber"
)

df_diarias = spark.table(
    "viagens_teste.silver.diarias"
)

Validação das fontes da camada Silver

In [0]:
print("Passagens:", df_passagens.count())
print("Hospedagens:", df_hospedagens.count())
print("Uber:", df_uber.count())
print("Diárias:", df_diarias.count())

Padronização de Passagens para a camada Gold

In [0]:
from pyspark.sql.functions import col

df_gold_passagens = df_passagens.select(
    col("id_solicitacao").alias("id_registro"),
    col("viajante"),
    col("data_viagem").alias("data_evento"),
    col("centro_custo"),
    col("diretoria"),
    col("valor_numerico").alias("valor"),
    col("tipo_despesa"),
    col("trimestre"),
    col("agencia"),
    col("situacao")
)

Validação da estrutura Gold de Passagens

In [0]:
df_gold_passagens.printSchema()

display(df_gold_passagens)

In [0]:
df_gold_passagens.count()

Padronização de Hospedagens para a camada Gold

In [0]:
df_gold_hospedagens = df_hospedagens.select(
    col("id_hospedagem").alias("id_registro"),
    col("viajante"),
    col("data_checkin").alias("data_evento"),
    col("centro_custo"),
    col("diretoria"),
    col("valor_numerico").alias("valor"),
    col("tipo_despesa"),
    col("trimestre"),
    col("agencia"),
    col("situacao")
)

Validação da estrutura Gold de Hospedagens

In [0]:
df_gold_hospedagens.printSchema()

print(
    df_gold_passagens.columns ==
    df_gold_hospedagens.columns
)

In [0]:
print(
    "Hospedagens:",
    df_gold_hospedagens.count()
)

Padronização de Uber para a camada Gold

In [0]:
from pyspark.sql.functions import col, lit

df_gold_uber = df_uber.select(
    col("id_corrida").alias("id_registro"),
    col("viajante"),
    col("data_corrida").alias("data_evento"),
    col("centro_custo"),
    col("diretoria"),
    col("valor_numerico").alias("valor"),
    col("tipo_despesa"),
    col("trimestre"),
    col("agencia"),
    lit(None).cast("string").alias("situacao")
)

Validação da estrutura Gold de Uber

In [0]:
df_gold_uber.printSchema()

print(
    df_gold_passagens.columns ==
    df_gold_uber.columns
)

In [0]:
print(
    "Uber:",
    df_gold_uber.count()
)

Padronização de Diárias para a camada Gold

In [0]:
df_gold_diarias = df_diarias.select(
    col("id_diaria").alias("id_registro"),
    col("viajante"),
    col("data_inicio").alias("data_evento"),
    col("centro_custo"),
    col("diretoria"),
    col("valor_numerico").alias("valor"),
    col("tipo_despesa"),
    col("trimestre"),
    col("agencia"),
    lit(None).cast("string").alias("situacao")
)

Validação da estrutura Gold de Diárias

In [0]:
df_gold_diarias.printSchema()

print(
    df_gold_passagens.columns ==
    df_gold_diarias.columns
)

In [0]:
print(
    "Diárias:",
    df_gold_diarias.count()
)

Consolidação da tabela fato de viagens

In [0]:
df_fato_viagens = (
    df_gold_passagens
    .unionByName(df_gold_hospedagens)
    .unionByName(df_gold_uber)
    .unionByName(df_gold_diarias)
)

In [0]:
print("Total da fato_viagens:", df_fato_viagens.count())

Validação da consolidação

In [0]:
df_fato_viagens.groupBy(
    "tipo_despesa"
).count().show()

In [0]:
from pyspark.sql.functions import sum

df_fato_viagens.groupBy(
    "tipo_despesa"
).agg(
    sum("valor").alias("valor_total")
).show()

Persistência da tabela fato na camada Gold

In [0]:
df_fato_viagens.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("viagens_teste.gold.fato_viagens")

In [0]:
df_fato_teste = spark.table(
    "viagens_teste.gold.fato_viagens"
)

display(df_fato_teste)

In [0]:
print("Registros salvos:", df_fato_teste.count())